# AI-Based Battery Health Monitoring System

This notebook loads the genuine NASA PCoE MATLAB archive, performs exploratory analysis, creates capacity-derived SOH, compares models and saves figures/metrics. Run after placing the NASA ZIP in `data/raw/`. No synthetic observations are substituted for the research data.

## 1. Dataset source and target
NASA PCoE Battery Data Set: https://phm-datasets.s3.amazonaws.com/NASA/5.+Battery+Data+Set.zip. SOH (%) = discharge capacity (Ah) / nominal capacity (2.0 Ah) × 100. SOC is a different quantity.

In [ ]:
from pathlib import Path
import sys, json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
ROOT=Path.cwd()
if not (ROOT/'src').exists(): ROOT=ROOT.parent
sys.path.insert(0,str(ROOT))
from src.data_loader import load_nasa_batteries
from src.config import FEATURES, FIGURES_DIR, METRICS_DIR
from src.preprocessing import clean_frame
FIGURES_DIR.mkdir(parents=True,exist_ok=True); METRICS_DIR.mkdir(parents=True,exist_ok=True)

## 2. Load and inspect actual dataset

In [ ]:
df=load_nasa_batteries(ROOT/'data'/'raw')
df=clean_frame(df)
print('Dimensions:',df.shape)
display(df.head())
display(df.dtypes.to_frame('dtype'))
display(df.describe(include='all').T)

## 3. Missing values and duplicate records

In [ ]:
display(df.isna().sum().to_frame('missing_count'))
print('Duplicate rows:',df.duplicated().sum())
print('Battery IDs:',df.battery_id.unique())

## 4. Distributions and correlation

In [ ]:
cols=['voltage_mean_v','current_mean_a','temperature_mean_c','soh_percent']
fig,axes=plt.subplots(2,2,figsize=(11,7))
for ax,col in zip(axes.flat,cols):
    sns.histplot(df[col].dropna(),kde=True,ax=ax); ax.set_title(col); ax.set_xlabel(col)
fig.tight_layout(); fig.savefig(FIGURES_DIR/'feature_distributions.png',dpi=160); plt.show()
plt.figure(figsize=(10,7)); sns.heatmap(df.select_dtypes('number').corr(),annot=True,cmap='coolwarm',fmt='.2f'); plt.title('Feature correlation'); plt.tight_layout(); plt.savefig(FIGURES_DIR/'correlation_heatmap.png',dpi=160); plt.show()

## 5. Parameter versus SOH and capacity degradation

In [ ]:
for col in ['voltage_mean_v','current_mean_a','temperature_mean_c']:
    plt.figure(figsize=(7,4)); sns.scatterplot(data=df,x=col,y='soh_percent',hue='battery_id'); plt.title(f'{col} vs SOH'); plt.tight_layout(); plt.savefig(FIGURES_DIR/f'{col}_vs_soh.png',dpi=160); plt.show()
plt.figure(figsize=(8,4)); sns.lineplot(data=df,x='cycle_index',y='discharge_capacity_ah',hue='battery_id'); plt.title('Discharge capacity across recorded cycles'); plt.ylabel('Capacity (Ah)'); plt.tight_layout(); plt.savefig(FIGURES_DIR/'capacity_by_cycle.png',dpi=160); plt.show()

## 6. Feature engineering and preprocessing
Cycle summaries are calculated by the loader. Capacity is excluded from predictors because it defines the target. Median imputation and scaling are fit inside the model pipeline using training data only.

In [ ]:
X=df[FEATURES]; y=df['soh_percent']
print('Features:',FEATURES)
print('Target range:',float(y.min()),float(y.max()))

## 7. Train and compare regression models

In [ ]:
from src.train_regression import train_regression
reg_results,reg_model,test_reg=train_regression(df)
print(json.dumps(reg_results,indent=2))

## 8. Train classification where supported

In [ ]:
from src.train_classification import train_classification
clf_results,clf_model,test_clf=train_classification(df)
print(json.dumps(clf_results,indent=2))

## 9. Evaluate and visualize held-out predictions

In [ ]:
pred_path=METRICS_DIR/'regression_test_predictions.csv'
pred=pd.read_csv(pred_path)
plt.figure(figsize=(6,6)); sns.scatterplot(data=pred,x='actual_soh',y='predicted_soh'); lo=min(pred.min()); hi=max(pred.max()); plt.plot([lo,hi],[lo,hi],'--'); plt.xlabel('Actual SOH (%)'); plt.ylabel('Predicted SOH (%)'); plt.title('Held-out battery: actual vs predicted'); plt.tight_layout(); plt.savefig(FIGURES_DIR/'actual_vs_predicted.png',dpi=160); plt.show()
residual=pred.actual_soh-pred.predicted_soh
plt.figure(figsize=(7,4)); sns.scatterplot(x=pred.predicted_soh,y=residual); plt.axhline(0,linestyle='--'); plt.xlabel('Predicted SOH (%)'); plt.ylabel('Residual (percentage points)'); plt.title('Regression residuals'); plt.tight_layout(); plt.savefig(FIGURES_DIR/'residuals.png',dpi=160); plt.show()

## 10. Interpretation and limitations
Review the JSON metrics and the held-out battery identity. The dataset contains few independent cells, so test results are uncertain. The classification labels are project thresholds, not safety standards. If a class is absent, do not manufacture examples. This model is not a certified BMS.